[Apache Parquet](https://parquet.apache.org/) has become a standard format for storing large analytical datasets due to its efficient columnar compression and encoding. However, many applications still require data to reside in a relational database like [MariaDB](https://mariadb.org/) for transactional workloads, reporting, or integration with existing systems. This notebook demonstrates how to bridge these two worlds by loading Parquet data directly into MariaDB.

The key to efficient ingestion is [ADBC (Arrow Database Connectivity)](https://arrow.apache.org/adbc/current/index.html), a database-agnostic API built on [Apache Arrow](https://arrow.apache.org/). Unlike traditional approaches that serialize data row-by-row, ADBC transfers data in columnar batches, significantly reducing overhead when moving large datasets. We'll use the [NYC Yellow Taxi trip dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) from January 2025—approximately 3.5 million records—to demonstrate how straightforward this process can be with just a few lines of Python.

Requirements:

- Python 3
- MariaDB or Docker

## Setup

Install the required dependencies:

In [1]:
%pip install -q dbc adbc-driver-manager pyarrow

Note: you may need to restart the kernel to use updated packages.


Install the MySQL ADBC driver:

In [ ]:
!dbc install -q mysql

If you don't already have a MariaDB instance running, start an instance with Docker:

In [3]:
!docker run -d --name some-mariadb -p 3306:3306 --env MARIADB_ROOT_PASSWORD=my-secret-pw mariadb

922119f7ce051780fe46af17ca2eef54c1a1a6115add134c665c462352c446af


Import dependencies:

In [2]:
import os
import urllib.request

import pyarrow.parquet as pq
from adbc_driver_manager import dbapi

Download a Parquet file and read it as a PyArrow Table:

In [4]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
file_path = "yellow_tripdata_2025-01.parquet"

urllib.request.urlretrieve(url=url, filename=file_path)
data = pq.read_table(file_path)

## Ingestion

Connect to MariaDB and ingest the data:

In [ ]:
with (
    dbapi.connect(
        driver="mysql",
        db_kwargs={"uri": "root:my-secret-pw@tcp(localhost:3306)/mysql"},
        autocommit=True,
    ) as connection,
    connection.cursor() as cursor,
):
    rows_inserted = cursor.adbc_ingest(table_name="yellow_tripdata", data=data)

print(f"Inserted {rows_inserted:,d} rows")

Inserted 3,475,226 rows


## Cleanup

Delete the downloaded Parquet file:

In [8]:
os.remove(file_path)

If you ran MariaDB with Docker earlier, stop the container:

In [9]:
!docker stop some-mariadb

some-mariadb
